# Previsao por Media Movel Simples por Slot (15 min)

Notebook dedicado ao modelo SMA por `slot_15m`, separado do fluxo principal para facilitar analise.


In [16]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


## Parametros

Ajuste os parametros abaixo para testar o modelo.


In [17]:
# =============================
# PERÍODO DE AVALIAÇÃO
# =============================

START_YEAR = 2024
START_MONTH = 8
END_YEAR = 2025
END_MONTH = 12

# None = todos os edifícios
BUILDING_FILTER = None

TARGET_COL = "consumption_kwh"
BUILDING_COL = "building"

INPUT_CSV = "https://raw.githubusercontent.com/lucasmedss/peca/main/cg_public_energy_consumption.csv"

SMA_SLOT_WINDOW = 21
SMA_SLOT_MIN_PERIODS = 14
SMA_TRAIN_LOOKBACK_DAYS = 21

USE_FIRST_WEEK = True
FIRST_WEEK_DAYS = 7

In [18]:
def load_history(
    dataframe: pd.DataFrame,
    target_col: str,
    building_col: str = "building",
    building_filter: str | None = None,
) -> pd.DataFrame:

    df = dataframe.copy()

    if building_col not in df.columns and "building_id" in df.columns:
        df[building_col] = df["building_id"]

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=["timestamp", target_col])

    if building_filter is not None:
        filtro = str(building_filter).strip()
        df = df[df[building_col].astype(str).str.strip() == filtro]

    history = (
        df.sort_values("timestamp")
        .set_index("timestamp")[target_col]
        .resample("15min")
        .sum(min_count=1)
        .to_frame(name=target_col)
        .dropna()
    )
    return history


def month_range(year: int, month: int) -> pd.DatetimeIndex:
    start = pd.Timestamp(year=year, month=month, day=1)
    end = (start + pd.offsets.MonthEnd(1)).normalize() + pd.Timedelta(hours=23, minutes=45)
    return pd.date_range(start=start, end=end, freq="15min")


def select_history_window(history: pd.DataFrame, lookback_days: int | None):
    if lookback_days is None:
        return history.copy()
    lookback_days = int(lookback_days)
    if lookback_days <= 0:
        return history.copy()
    cutoff = history.index.max() - pd.Timedelta(days=lookback_days)
    return history.loc[history.index >= cutoff].copy()


In [19]:
def forecast_idx_sma_slot(
    future_idx: pd.DatetimeIndex,
    slot_predictions: pd.Series,
    fallback: float
) -> pd.DataFrame:
    """Projeta consumo por timestamp futuro usando previsão por slot_15m."""

    future = pd.DataFrame(index=future_idx)
    future["slot_15m"] = future.index.hour * 4 + (future.index.minute // 15)

    future["consumption_kwh_pred_sma_slot"] = (
        future["slot_15m"]
        .map(slot_predictions)
        .fillna(fallback)
        .astype(float)
    )

    return future[["consumption_kwh_pred_sma_slot"]]

In [20]:
# =============================
# GERAR RANGE DE MESES DO PERÍODO
# =============================

start_period = pd.Timestamp(year=START_YEAR, month=START_MONTH, day=1)
end_period = pd.Timestamp(year=END_YEAR, month=END_MONTH, day=1)

months = pd.date_range(start=start_period, end=end_period, freq="MS")

all_forecasts = []

for month_start in months:

    year = month_start.year
    month = month_start.month

    month_full = month_range(year, month)

    # Se estiver usando primeira semana real
    if USE_FIRST_WEEK:
        first_week_end = month_full.min() + pd.Timedelta(days=FIRST_WEEK_DAYS)
        future_idx = month_full[month_full >= first_week_end]
    else:
        future_idx = month_full

    forecast_sma_slot = forecast_idx_sma_slot(
        future_idx,
        next_pred_per_slot,
        fallback_mean
    )

    forecast_sma_slot["year"] = year
    forecast_sma_slot["month"] = month

    all_forecasts.append(forecast_sma_slot)

# Concatenar tudo
forecast_sma_slot = pd.concat(all_forecasts)

print(f"Período previsto: {forecast_sma_slot.index.min()} até {forecast_sma_slot.index.max()}")

display(forecast_sma_slot.head())

Período previsto: 2024-08-08 00:00:00 até 2025-12-31 23:45:00
Total previsto no período (kWh): 16,926.09


,consumption_kwh_pred_sma_slot,year,month
2024-08-08 00:00:00,0.322317,2024,8
2024-08-08 00:15:00,0.324479,2024,8
2024-08-08 00:30:00,0.322410,2024,8
2024-08-08 00:45:00,0.323652,2024,8
2024-08-08 01:00:00,0.323512,2024,8


In [21]:
# =============================
# AVALIAÇÃO MULTI-ANO / MULTI-PRÉDIO
# =============================

df_raw = pd.read_csv(INPUT_CSV, parse_dates=["timestamp"])

# Lista de edifícios automaticamente
buildings = (
    df_raw[BUILDING_COL]
    .dropna()
    .astype(str)
    .unique()
)

resultados = []

periodo_inicio = pd.Timestamp(year=START_YEAR, month=START_MONTH, day=1)
periodo_fim = pd.Timestamp(year=END_YEAR, month=END_MONTH, day=1)

periodos = pd.date_range(
    start=periodo_inicio,
    end=periodo_fim,
    freq="MS"
)

for building in buildings:

    history = load_history(
        df_raw,
        TARGET_COL,
        BUILDING_COL,
        building
    )

    if history.empty:
        continue

    for periodo in periodos:

        forecast_start = periodo

        history_until_month = history.loc[
            history.index < forecast_start
        ]

        hist_slot = select_history_window(
            history_until_month,
            SMA_TRAIN_LOOKBACK_DAYS
        )

        if hist_slot.empty:
            continue

        hist_slot = hist_slot.copy()

        hist_slot["slot_15m"] = (
            hist_slot.index.hour * 4
            + (hist_slot.index.minute // 15)
        )

        hist_slot["sma_slot"] = hist_slot.groupby("slot_15m")[TARGET_COL].transform(
            lambda s: s.rolling(
                window=SMA_SLOT_WINDOW,
                min_periods=SMA_SLOT_MIN_PERIODS
            ).mean()
        )

        next_pred_per_slot = hist_slot.groupby("slot_15m")["sma_slot"].last()

        fallback_mean = (
            float(hist_slot["sma_slot"].dropna().mean())
            if hist_slot["sma_slot"].notna().any()
            else float(hist_slot[TARGET_COL].mean())
        )

        future_idx = month_range(periodo.year, periodo.month)

        forecast = forecast_idx_sma_slot(
            future_idx,
            next_pred_per_slot,
            fallback_mean
        )

        previsto_mes = forecast["consumption_kwh_pred_sma_slot"].sum()

        actual = history.loc[
            future_idx.min():future_idx.max(),
            [TARGET_COL]
        ]

        real_mes = (
            actual[TARGET_COL].sum()
            if not actual.empty else None
        )

        if real_mes is None:
            continue

        erro = previsto_mes - real_mes

        smape = (
            100 * abs(erro) /
            ((abs(real_mes) + abs(previsto_mes)) / 2)
            if (real_mes + previsto_mes) != 0 else None
        )

        resultados.append({
            "Building": building,
            "Ano": periodo.year,
            "Mes": periodo.month,
            "Periodo": periodo.strftime("%Y-%m"),
            "Real": real_mes,
            "Previsto": previsto_mes,
            "Erro_kWh": erro,
            "Erro_%": (
        (erro / real_mes) * 100
        if real_mes != 0 else None
    )
        })

df_sma_compare = pd.DataFrame(resultados)

display(df_sma_compare.round(2))

,Building,Ano,Mes,Periodo,Real,Previsto,Erro_kWh,Erro_%
0,ESCOLA_MUNICIPAL,2024,8,2024-08,1389.08,1315.06,-74.02,-5.33
1,ESCOLA_MUNICIPAL,2024,9,2024-09,1288.25,1372.60,84.35,6.55
2,ESCOLA_MUNICIPAL,2024,10,2024-10,1334.34,1343.67,9.32,0.70
3,ESCOLA_MUNICIPAL,2024,11,2024-11,1363.78,1318.43,-45.34,-3.32
4,ESCOLA_MUNICIPAL,2024,12,2024-12,1113.31,1368.52,255.21,22.92
...,...,...,...,...,...,...,...,...
63,TEATRO_MUNICIPAL,2025,8,2025-08,9015.33,8387.15,-628.18,-6.97
64,TEATRO_MUNICIPAL,2025,9,2025-09,8151.56,10744.80,2593.24,31.81
65,TEATRO_MUNICIPAL,2025,10,2025-10,6474.29,6728.61,254.32,3.93
66,TEATRO_MUNICIPAL,2025,11,2025-11,7565.71,6106.82,-1458.88,-19.28


In [22]:
metricas_por_predio = df_sma_compare.groupby("Building").apply(
    lambda x: pd.Series({
        "RMSE": np.sqrt(np.mean(x["Erro_kWh"]**2)),
        "SMAPE_%": np.mean(
            100 * np.abs(x["Previsto"] - x["Real"]) /
            ((np.abs(x["Real"]) + np.abs(x["Previsto"])) / 2)
        )
    })
).reset_index()

display(metricas_por_predio.round(2))

/tmp/ipykernel_364/3603022968.py:1: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,Building,RMSE,SMAPE_%
0,ESCOLA_MUNICIPAL,185.01,9.93
1,SAMU,939.09,10.31
2,TEATRO_MUNICIPAL,2934.99,39.28
3,UPA,1470.12,8.14


In [23]:
# =============================
# HISTÓRICO SOMENTE ESCOLA
# =============================

BUILDING_FILTER = "ESCOLA_MUNICIPAL"

df_raw = pd.read_csv(INPUT_CSV, parse_dates=["timestamp"])

history = load_history(
    df_raw,
    TARGET_COL,
    BUILDING_COL,
    BUILDING_FILTER
)

print(f"Edifício analisado: {BUILDING_FILTER}")
print(f"Período disponível: {history.index.min()} até {history.index.max()}")

Edifício analisado: ESCOLA_MUNICIPAL
Período disponível: 2023-01-01 00:00:00 até 2025-12-31 23:45:00


In [24]:
# =====================================================
# MÉTRICAS GLOBAIS — ESCOLA MUNICIPAL
# =====================================================

df_valid = df_sazonal_multi.copy()

# Remover possíveis valores nulos
df_valid = df_valid.dropna()

# Erro
df_valid["Erro"] = df_valid["Previsto_Sazonal"] - df_valid["Real"]

# RMSE
rmse = np.sqrt(np.mean(df_valid["Erro"]**2))

# MAE
mae = np.mean(np.abs(df_valid["Erro"]))

# sMAPE (robusto)
smape = np.mean(
    2*np.abs(df_valid["Erro"]) /
    (np.abs(df_valid["Real"]) + np.abs(df_valid["Previsto_Sazonal"]))
) * 100

# Erro percentual médio
erro_medio_percentual = np.mean(
    df_valid["Erro"] / df_valid["Real"]
) * 100


print("\n===== MÉTRICAS MODELO SAZONAL — ESCOLA MUNICIPAL =====\n")
print(f"Período: Ago/2024 até Dez/2025\n")
print(f"RMSE  : {rmse:.2f} kWh")
print(f"MAE   : {mae:.2f} kWh")
print(f"sMAPE : {smape:.2f} %")
print(f"Erro Médio (%) : {erro_medio_percentual:.2f} %")

display(df_valid.round(2))


===== MÉTRICAS MODELO SAZONAL — ESCOLA MUNICIPAL =====

Período: Ago/2024 até Dez/2025

RMSE  : 74.59 kWh
MAE   : 61.34 kWh
sMAPE : 4.81 %
Erro Médio (%) : 0.60 %


,Ano,Mes,Periodo,Real,Previsto_SMA,Previsto_Sazonal,Erro
0,2024,8,2024-08,1389.08,1315.06,1315.06,-74.02
1,2024,9,2024-09,1288.25,1372.60,1372.60,84.35
2,2024,10,2024-10,1334.34,1343.67,1343.67,9.32
3,2024,11,2024-11,1363.78,1318.43,1318.43,-45.34
4,2024,12,2024-12,1113.31,1368.52,1234.35,121.04
5,2025,1,2025-01,949.52,1020.46,1020.46,70.94
6,2025,2,2025-02,1324.74,925.77,1148.00,-176.74
7,2025,3,2025-03,1604.51,1610.84,1610.84,6.33
8,2025,4,2025-04,1668.56,1687.68,1687.68,19.11
9,2025,5,2025-05,1711.78,1714.33,1714.33,2.55


In [25]:
# =====================================================
# ESTATÍSTICAS MODELO SAZONAL
# Período: Ago/2024 até Dez/2025
# =====================================================
MESES_SAZONAIS = {2, 6, 7, 12}

START_EVAL = pd.Timestamp("2024-08-01")
END_EVAL   = pd.Timestamp("2025-12-31")

anos_avaliacao = [2024, 2025]

resultados_multi = []

for ano in anos_avaliacao:

    for mes in range(1, 13):

        periodo_mes = pd.Timestamp(year=ano, month=mes, day=1)

        if periodo_mes < START_EVAL or periodo_mes > END_EVAL:
            continue

        future_idx = month_range(ano, mes)

        actual_full = history.loc[
            future_idx.min():future_idx.max(),
            [TARGET_COL]
        ]

        if actual_full.empty:
            continue

        real_mes = actual_full[TARGET_COL].sum()

        # =========================
        # SMA NORMAL
        # =========================

        history_until_month = history.loc[
            history.index < periodo_mes
        ]

        hist_slot = select_history_window(
            history_until_month,
            SMA_TRAIN_LOOKBACK_DAYS
        )

        if hist_slot.empty:
            continue

        hist_slot = hist_slot.copy()
        hist_slot["slot_15m"] = (
            hist_slot.index.hour * 4 +
            (hist_slot.index.minute // 15)
        )

        hist_slot["sma_slot"] = hist_slot.groupby("slot_15m")[TARGET_COL].transform(
            lambda s: s.rolling(
                window=SMA_SLOT_WINDOW,
                min_periods=SMA_SLOT_MIN_PERIODS
            ).mean()
        )

        next_pred = hist_slot.groupby("slot_15m")["sma_slot"].last()
        fallback = float(hist_slot[TARGET_COL].mean())

        forecast_sma = forecast_idx_sma_slot(
            future_idx,
            next_pred,
            fallback
        )

        previsto_sma = forecast_sma["consumption_kwh_pred_sma_slot"].sum()

        # =========================
        # MODELO SAZONAL AJUSTADO
        # =========================

        if mes in MESES_SAZONAIS:

            first_week_end = future_idx.min() + pd.Timedelta(days=FIRST_WEEK_DAYS)

            real_first_week = history.loc[
                future_idx.min():first_week_end,
                [TARGET_COL]
            ]

            consumo_primeira_semana = (
                real_first_week[TARGET_COL].sum()
                if not real_first_week.empty else 0
            )

            restante_idx = future_idx[future_idx >= first_week_end]

            base_idx = month_range(ano - 1, mes)

            base_data = history.loc[
                base_idx.min():base_idx.max(),
                [TARGET_COL]
            ]

            if not base_data.empty:

                base_data = base_data.copy()
                base_data["slot_15m"] = (
                    base_data.index.hour * 4 +
                    (base_data.index.minute // 15)
                )

                perfil_mes = base_data.groupby("slot_15m")[TARGET_COL].mean()

                forecast_temp = pd.DataFrame(index=restante_idx)
                forecast_temp["slot_15m"] = (
                    forecast_temp.index.hour * 4 +
                    (forecast_temp.index.minute // 15)
                )

                forecast_temp["pred"] = (
                    forecast_temp["slot_15m"]
                    .map(perfil_mes)
                    .fillna(perfil_mes.mean())
                )

                consumo_restante = forecast_temp["pred"].sum()

            else:
                consumo_restante = 0

            previsto_sazonal = consumo_primeira_semana + consumo_restante

        else:
            previsto_sazonal = previsto_sma

        resultados_multi.append({
            "Ano": ano,
            "Mes": mes,
            "Periodo": periodo_mes.strftime("%Y-%m"),
            "Real": real_mes,
            "Previsto_SMA": previsto_sma,
            "Previsto_Sazonal": previsto_sazonal
        })

df_sazonal_multi = pd.DataFrame(resultados_multi)

# =====================================================
# MÉTRICAS GLOBAIS
# =====================================================

rmse = np.sqrt(
    np.mean((df_sazonal_multi["Real"] - df_sazonal_multi["Previsto_Sazonal"])**2)
)

smape = np.mean(
    2*np.abs(df_sazonal_multi["Real"] - df_sazonal_multi["Previsto_Sazonal"]) /
    (np.abs(df_sazonal_multi["Real"]) + np.abs(df_sazonal_multi["Previsto_Sazonal"]))
) * 100


print("\n===== MÉTRICAS MODELO SAZONAL (Ago/2024–Dez/2025) =====\n")
print(f"RMSE  : {rmse:.2f} kWh")
print(f"sMAPE : {smape:.2f} %")

display(df_sazonal_multi.round(2))


===== MÉTRICAS MODELO SAZONAL (Ago/2024–Dez/2025) =====

RMSE  : 74.59 kWh
sMAPE : 4.81 %


,Ano,Mes,Periodo,Real,Previsto_SMA,Previsto_Sazonal
0,2024,8,2024-08,1389.08,1315.06,1315.06
1,2024,9,2024-09,1288.25,1372.60,1372.60
2,2024,10,2024-10,1334.34,1343.67,1343.67
3,2024,11,2024-11,1363.78,1318.43,1318.43
4,2024,12,2024-12,1113.31,1368.52,1234.35
5,2025,1,2025-01,949.52,1020.46,1020.46
6,2025,2,2025-02,1324.74,925.77,1148.00
7,2025,3,2025-03,1604.51,1610.84,1610.84
8,2025,4,2025-04,1668.56,1687.68,1687.68
9,2025,5,2025-05,1711.78,1714.33,1714.33


In [26]:
import plotly.graph_objects as go
import pandas as pd

df_plot = df_sazonal_multi.copy()
df_plot["Data"] = pd.to_datetime(df_plot["Periodo"])
df_plot = df_plot.sort_values("Data")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["Data"],
    y=df_plot["Real"],
    mode="lines+markers",
    name="Consumo Real",
    line=dict(width=3)
))

fig.add_trace(go.Scatter(
    x=df_plot["Data"],
    y=df_plot["Previsto_SMA"],
    mode="lines+markers",
    name="SMA Original",
    line=dict(dash="dot", width=3)
))

fig.add_trace(go.Scatter(
    x=df_plot["Data"],
    y=df_plot["Previsto_Sazonal"],
    mode="lines+markers",
    name="SMA com Sazonalidade",
    line=dict(dash="dash", width=3)
))

fig.update_layout(
    title="",
    xaxis_title="Período",
    yaxis_title="Consumo Total Mensal (kWh)",
    plot_bgcolor="white",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        y=1.1,
        x=0.5,
        xanchor="center"
    )
)

fig.update_yaxes(gridcolor="whitesmoke", tickformat=",")

MESES_PT = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
    5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Set", 10: "Out", 11: "Nov", 12: "Dez"
}

# Criar rótulo apenas com mês em português
df_plot["Mes_PT"] = df_plot["Data"].dt.month.map(MESES_PT)

fig.update_xaxes(
    showgrid=False,
    tickmode="array",
    tickvals=df_plot["Data"],
    ticktext=df_plot["Mes_PT"],
    tickangle=0
)

fig.show()